# Multi-cell runbook - Kaggle (Step B, sharded + dissociation)

Spatial generalization: pooled in-domain, leave-one-cell-out **transfer** (run as one short job per
held-out cell), regional >=30% alert. Independent of the single-cell run. Results identical to running
it all at once. Aligned with Methods (REV3).

## Before running (Settings panel)
1. **Accelerator: GPU T4 x2** (or P100). Without GPU -> RF + baselines only.
2. **Persistence: Files only** (so /kaggle/working + parts/ survive between interactive sessions).
3. **Internet: ON** only if you let it download (you can attach your data instead).
4. **Add Data**: your **scripts** Dataset (must be the UPDATED 8 .py incl analyze_dissociation.py)
   and your **data** Dataset (cells.csv + 10 cell_*.csv). For RESUME, also add this notebook's
   previous Version output.


## Setup - copy scripts, resume previous parts, check runtime

In [ ]:
import os, sys, glob, shutil, urllib.request
hits = glob.glob('/kaggle/input/**/tcn_drought_multicell.py', recursive=True)
assert hits, "Attach your (updated) scripts Dataset via 'Add Data'."
SCRIPTS_DIR = os.path.dirname(hits[0]); print('scripts:', SCRIPTS_DIR)
for f in glob.glob(os.path.join(SCRIPTS_DIR, '*.py')): shutil.copy(f, '/kaggle/working/')
os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')

import tcn_drought_multicell as _M
assert 'phase' in _M.CONFIG_MC and hasattr(_M, 'merge_shards'), \
    "STOP: tcn_drought_multicell.py BUKAN versi sharded. Perbarui scripts dataset (New Version)."
print('OK: versi sharded terdeteksi.')

RESULTS_MC = '/kaggle/working/results_mc'
# RESUME: carry forward transfer parts from a previous Version output, if attached
for prev in glob.glob('/kaggle/input/**/results_mc/parts', recursive=True):
    dst = os.path.join(RESULTS_MC, 'parts'); os.makedirs(dst, exist_ok=True)
    for f in glob.glob(prev + '/*.csv'): shutil.copy(f, dst)
    print('resumed parts from', prev)
try:
    import tensorflow as tf; print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
except Exception as e: print('No TF -> RF only:', e)

## Data - attached data Dataset, else download

In [ ]:
import pandas as pd
cand = glob.glob('/kaggle/input/**/cells.csv', recursive=True)
DATA_DIR = os.path.dirname(cand[0]) if cand else 'data'
if not glob.glob(os.path.join(DATA_DIR, 'cell_*.csv')):
    !python download_nasapower_cells.py
    DATA_DIR = 'data'
print('DATA_DIR =', DATA_DIR)
print(pd.read_csv(os.path.join(DATA_DIR, 'cells.csv')).to_string(index=False))
print('\nWS2M in EVERY cell:',
      all('WS2M' in pd.read_csv(f, nrows=1).columns for f in glob.glob(os.path.join(DATA_DIR, 'cell_*.csv'))))

## Step B1 - MEASURE one transfer cell (decide the model set & estimate time)

Defaults: `transfer_models=['TCN','RandomForest']`, `seeds=[0,1,2]`, all 4 horizons. If one cell is too
slow for your budget, switch to **RF-first** (uncomment the light line) in BOTH this cell and the loop.

In [ ]:
import time, importlib, tcn_drought_multicell as M; importlib.reload(M)
t = time.time()
c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC
c['phase'] = 'transfer'; c['only_held'] = 'lombok'
# LIGHT (RF-first; keep identical in the loop below if you use it):
# c['transfer_models'] = ['RandomForest']
M.main(c)
print(f'\nONE transfer cell ~ {(time.time()-t)/60:.1f} min  ->  x10 cells + in-domain = your total')

## Step B2 - LOOP the remaining transfer cells (skips ones already saved)

In [ ]:
import importlib, tcn_drought_multicell as M; importlib.reload(M)
CELLS = ['lombok','sumbawaB','sumbawaC','bima','manggarai','ende','floresT','sumba','kupang','soe']
done = {os.path.basename(p).split('__')[1] for p in glob.glob(os.path.join(RESULTS_MC,'parts','transfer__*'))}
for held in CELLS:
    if held in done:
        print('skip (done):', held); continue
    c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC
    c['phase'] = 'transfer'; c['only_held'] = held
    # c['transfer_models'] = ['RandomForest']   # <- use the SAME light setting as B1 if you chose it
    M.main(c)

## Step B3 - IN-DOMAIN + regional alert (light; whole)

In [ ]:
import importlib, tcn_drought_multicell as M; importlib.reload(M)
c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC; c['phase'] = 'indomain'
M.main(c)

## Step B4 - MERGE -> metrics_transfer.csv + skill_map.csv

In [ ]:
import importlib, tcn_drought_multicell as M; importlib.reload(M)
c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC; c['phase'] = 'merge'
M.main(c)
print('\nresults_mc files:', sorted(f for f in os.listdir(RESULTS_MC) if f.endswith('.csv')))

## Step B5 - dissociation at the regional scale (no model re-run)
Same Section-2.14 analysis as single-cell, applied to the multi-cell pooled and regional DSS.

In [ ]:
!python analyze_dissociation.py {RESULTS_MC}/metrics_indomain.csv {RESULTS_MC}/dss_multicell.csv
print('--- regional (>=30% rule) ---')
!python analyze_dissociation.py {RESULTS_MC}/metrics_indomain.csv {RESULTS_MC}/dss_regional.csv

## Step B6 - verify + zip for download

In [ ]:
import os, shutil
need = ['metrics_indomain.csv','metrics_transfer.csv','skill_map.csv','dss_multicell.csv','dss_regional.csv']
have = set(os.listdir(RESULTS_MC))
for f in need: print(('OK   ' if f in have else 'MISSING ') + f)
nparts = len(glob.glob(os.path.join(RESULTS_MC,'parts','transfer__*')))
print(f'transfer parts present: {nparts}/10')
shutil.make_archive(RESULTS_MC, 'zip', RESULTS_MC); print('zipped results_mc.zip -> Output panel')